# HFE, HFE-Dragon, HFE-IP and HFE-IP-Dragon: a companion notebook

This notebook is a compact, self-contained SageMath implementation of a
family of multivariate signature schemes built on **Hidden Field Equations
(HFE)**:

1. **Plain HFE** — the classical Patarin construction: a low-degree secret
   univariate polynomial $F$ over $\mathbb{F}_{q^n}$ is expressed, via a
   $\mathbb{F}_q$-linear change of basis, as a system of $n$ quadratic
   polynomials in $n$ variables over $\mathbb{F}_q$; the public key hides
   this structure behind two secret invertible linear maps $S$ and $T$
   (with $T$ additionally *projected* from $n$ down to $m \le n$ equations).
2. **HFE-Dragon** — the secret polynomial is extended with extra terms that
   couple $X$ linearly to hash-derived auxiliary variables $Y_1,\dots,Y_k$,
   which are appended to the public variable set.
3. **HFE-IP** ("internal perturbation") — a small number $r$ of hidden
   "vinegar-like" variables $z = M_Z\, x$ are folded into $F$; the signer
   recovers a valid signature by guessing $z$ until the guess is consistent
   with the recovered preimage.
4. **HFE-IP-Dragon** — the composition of both add-ons.

**Scope of this notebook.** It is a *proof-of-concept companion to a paper*,
not a production implementation: there is no side-channel protection, the
message hash is a toy SHA-256-based digit expansion, and parameters are
chosen purely to demonstrate correctness quickly. Its purpose is to make the
constructions above precise and easy to check against the paper, not to be
a competitive or secure implementation.

**Base field.** Every construction below is written directly in terms of a
prime $q$, and the demonstration section instantiates the same four schemes
both for $q = 2$ and for a prime $q > 2$. The only genuine mathematical
changes needed to support $q > 2$ (beyond replacing `GF(2)` by `GF(q)`
and bit-shifts `1 << k` by `q^k`) are noted inline where they occur: a few
places in the original code implicitly used the fact that, over $\mathbb F_2$,
a coordinate is either $0$ or $1$, so "add a coefficient whenever a bit is
set" is the same as "scale the coefficient by that bit". Over $\mathbb F_q$
this is no longer true and the coefficient must be *multiplied* by the
actual field element; this notebook does the multiplication explicitly
everywhere, which reduces to the original behaviour exactly when $q = 2$.

## Layout

1. Finite-field encoding utilities
2. HFE polynomials and their quadratic-form representation
3. Add-on transformations (Dragon, Internal Perturbation)
4. Public parameters
5. Scheme 1 — plain HFE
6. Scheme 2 — HFE-Dragon
7. Scheme 3 — HFE-IP
8. Scheme 4 — HFE-IP-Dragon
9. Optional: exporting the MQ system as explicit polynomial equations
10. Demonstration ($q=2$ and $q$ prime)


In [35]:
reset()
%display latex

import hashlib
from collections import namedtuple

## 1. Finite-field encoding utilities

Throughout, $K = \mathbb{F}_{q^n}$ is the *secret* working field and
$K_{\mathrm{pub}} = \mathbb{F}_{q^m}$ is the *public* field the verification
equation is stated over ($m \le n$). Coefficients of $K$ are repeatedly
"unpacked" into their $n$ coordinates over the prime field $\mathbb F_q$
(w.r.t. the power basis $1, \alpha, \dots, \alpha^{n-1}$), passed through a
public $\mathbb F_q$-linear projection $T_{\mathrm{pub}} \in \mathbb
F_q^{m \times n}$, and "repacked" as an element of $K_{\mathrm{pub}}$. The
two helpers below implement exactly that, and everything else in the
notebook (masking, projecting the quadratic form, verifying a signature) is
built on top of them.

In [36]:
def field_element_digits(elt, length):
    """
    GF(q)-coordinates of a finite-field element `elt`, expressed in the
    power basis 1, a, a^2, ... of its field, zero-padded/truncated to
    exactly `length` entries.

    Returns a plain Python list of ints in [0, q).
    """
    coeffs = [int(c) for c in elt.polynomial().list()]
    coeffs += [0] * (length - len(coeffs))
    return coeffs[:length]


def hash_to_digits(message, length, q):
    """
    Deterministically hash `message` (any object with a stable `repr`) into
    `length` base-q digits in [0, q-1], derived from a SHA-256 digest.

    This stands in for a proper domain-separated hash-to-(Z/qZ)^length
    function. It is adequate for a proof-of-concept but should not be used
    as-is in a real implementation.
    """
    digest = int(hashlib.sha256(repr(message).encode()).hexdigest(), 16)
    digits = []
    for _ in range(length):
        digits.append(digest % q)
        digest //= q
    return digits


def evaluate_affine_form(Q, L, cst, x):
    """
    Evaluate x^T Q x + L.x + cst, where Q is a square matrix and L a
    vector over some field K, cst in K, and x is any length-matching
    iterable of elements coercible into K (e.g. a signature, i.e. a
    vector over the base field GF(q)).

    Note the explicit *linear* part L: over GF(2), x_i^2 = x_i, so a
    linear term can be folded into the diagonal of a "quadratic" matrix
    at no cost -- a classic trick that only holds in characteristic 2.
    For q > 2 that identity is false (e.g. 2^2 = 1 != 2 in GF(3)), so
    linear and quadratic contributions must be tracked separately; this
    is exactly the difference between this notebook's affine-quadratic
    representation and the purely-quadratic one that a q=2-only
    implementation can get away with.
    """
    K = Q.base_ring()
    xK = vector(K, list(x))
    Lv = vector(K, list(L))
    return xK * Q * xK + Lv * xK + cst


def random_invertible_matrix(field, dim):
    """A uniformly random invertible dim x dim matrix over `field`."""
    return random_matrix(field, dim, algorithm="echelonizable", rank=dim)


def encode_public_coefficient(coeff, Tpub, q, n, Kpub):
    """
    Re-encode a coefficient `coeff` living in the secret field K = GF(q^n)
    as an element of the public field Kpub = GF(q^m): decompose `coeff`
    into its n GF(q)-digits, apply the public projection matrix Tpub
    (an m x n matrix over GF(q), i.e. the first m rows of the secret map T),
    and repack the resulting m digits as an element of Kpub.
    """
    digits = vector(GF(q), field_element_digits(coeff, n))
    projected = Tpub * digits
    Rm = Kpub.polynomial_ring()
    return Kpub(Rm(list(projected)))


def public_affine_form(Q, L, S, T, q, n, m, Kpub):
    """
    Build the public affine-quadratic form from a secret quadratic form Q
    (an N x N matrix over K = GF(q^n)), a secret linear part L (length N
    over K), and the secret linear maps S (N x N over GF(q), masking the
    variables) and T (n x n over GF(q), whose first m rows project the n
    equations down to m public equations). N is either n or n +
    hash_length depending on the scheme.

    Substituting x = y S^{-1}... (equivalently: masking by S) gives

        Q_S = S Q S^T                (new quadratic part)
        L_S = L S^T                  (new linear part)

    and each entry is then projected to the public field via pi(.) =
    `encode_public_coefficient` (using Tpub = T[:m, :]), exactly as for
    the plain-HFE constant term. Only the upper triangle of Q_pub is
    filled (see `encode_public_coefficient`'s docstring / the notebook
    text for why folding is valid for any q).

    Returns (Qpub, Lpub).
    """
    N = Q.nrows()
    K = Q.base_ring()
    S_K = S.change_ring(K)
    QS = S_K * Q * S_K.transpose()
    LS = vector(K, list(L)) * S_K.transpose()
    Tpub = T[:m, :]

    Qpub = Matrix(Kpub, N, N)
    for i in range(N):
        for j in range(i, N):
            coeff = QS[i, i] if i == j else QS[i, j] + QS[j, i]
            Qpub[i, j] = encode_public_coefficient(coeff, Tpub, q, n, Kpub)

    Lpub = vector(Kpub, [encode_public_coefficient(LS[i], Tpub, q, n, Kpub)
                          for i in range(N)])
    return Qpub, Lpub

## 2. HFE polynomials and their quadratic-form representation

An HFE polynomial of shape $(I, J)$ has $q$-degree $q^I + q^J$:

$$
F(X) = X^{q^I+q^J}
     + \sum_{i<I,\ j\le J} c_{i,j} X^{q^i+q^j}
     + \sum_{i\le I} c_i X^{q^i}
     + c_0 ,
\qquad c_{i,j}, c_i, c_0 \in K = \mathbb F_{q^n}.
$$

Writing $X = \sum_{k} \theta_k x_k$ with $\theta_k = \alpha^k$ ($\alpha$ a
generator of $K$) and using that $x_k^{q^t} = x_k$ for $x_k \in \mathbb F_q$
(so the Frobenius map is $\mathbb F_q$-linear on the $x_k$), $F(X) - c_0$
expands into a genuine quadratic form
$\sum_{i \le j} Q[i,j]\, x_i x_j$ over $K$ in the $n$ variables $x_0,
\dots,x_{n-1}$. `hfe_quadratic_form` builds this matrix $Q$ directly from
the coefficients of $F$.

In [37]:
def random_hfe_polynomial(Rx, q, HFEDegI, HFEDegJ):
    """
    Draw a uniformly random HFE polynomial of shape (HFEDegI, HFEDegJ):

        F(X) = X^(q^HFEDegI + q^HFEDegJ)
             + sum_{i<HFEDegI, j<=HFEDegJ} c_{i,j} X^(q^i + q^j)
             + sum_{i<=HFEDegI}            c_i     X^(q^i)
             + c_0

    with coefficients drawn uniformly from K = Rx.base_ring().
    """
    K = Rx.base_ring()
    X = Rx.gen()
    HFEDeg = q**HFEDegI + q**HFEDegJ
    F = X**HFEDeg
    F += sum(K.random_element() * X**(q**i + q**j)
              for j in range(HFEDegJ + 1) for i in range(HFEDegI))
    F += sum(K.random_element() * X**(q**i) for i in range(HFEDegI + 1))
    return F


def hfe_quadratic_form(F, q, n, HFEDegI, HFEDegJ):
    """
    Build the pair (Q, L) over K = F.base_ring() = GF(q^n) associated with
    the HFE polynomial F of shape (HFEDegI, HFEDegJ), i.e. the n x n
    quadratic part Q and the length-n linear part L such that

        F(X) - F(0)  =  sum_{i,j} Q[i,j] x_i x_j  +  sum_i L[i] x_i    over K,

    once X = sum_k theta_k x_k is substituted (theta_k = alpha^k). This
    uses two Frobenius-type identities, both valid for *any* prime q (not
    just q = 2): x_k^(q^t) = x_k for x_k in GF(q), which makes every
    "single q-power" term X^(q^t) expand as a genuinely *linear*
    combination sum_k theta_k^(q^t) x_k (so it belongs in L); and the
    ordinary binomial expansion of (X^(q^i))^2, needed for the "doubled"
    terms X^(2 q^i) that appear when the two exponents making up a
    quadratic HFE term coincide (i = j) -- these are genuinely *quadratic*
    (they involve x_k^2, not x_k) and belong in Q.

    A q = 2-only implementation can get away with folding every term into
    a single "quadratic" matrix, because x_k^2 = x_k for x_k in GF(2)
    collapses the two cases into one; this is exactly the corner the
    original code cut (see the notebook introduction).
    """
    K = F.base_ring()
    alpha = K.gen()
    theta = [alpha**i for i in range(n)]
    Q = Matrix(K, n, n)
    L = vector(K, n)

    # --- X^1 term: genuinely LINEAR (exponent q^0 = 1) ---------------------
    if HFEDegI == 0 and HFEDegJ == 0:
        for i in range(n):
            L[i] += theta[i]
    else:
        c = F[1]
        for i in range(n):
            L[i] += theta[i] * c

    # --- remaining terms, one HFE "level" k = 0 .. HFEDegI-1 at a time ----
    for k in range(HFEDegI):
        exp_i = q**k

        # quadratic cross terms X^(q^k) X^(q^l), l < k (genuinely quadratic,
        # since the two exponents q^k, q^l differ)
        for l in range(k):
            exp_j = q**l
            coef = F[exp_i + exp_j]
            for i in range(n):
                ti = theta[i] ** exp_i
                for j in range(n):
                    tj = theta[j] ** exp_j
                    Q[i, j] += ti * tj * coef

        # linear term X^(q^(k+1)): genuinely LINEAR
        exp = q**(k + 1)
        coef = F[exp]
        for i in range(n):
            L[i] += (theta[i] ** exp) * coef

    # --- highest quadratic-degree cross terms (genuinely quadratic) -------
    exp_i = q**HFEDegI
    for l in range(HFEDegJ):
        exp_j = q**l
        coef = F[exp_i + exp_j]
        for i in range(n):
            ti = theta[i] ** exp_i
            for j in range(n):
                tj = theta[j] ** exp_j
                Q[i, j] += ti * tj * coef

    # --- leading (monic) term -----------------------------------------------
    if HFEDegI != HFEDegJ:
        exp_j = q**HFEDegJ
        for i in range(n):
            ti = theta[i] ** exp_i
            for j in range(n):
                tj = theta[j] ** exp_j
                Q[i, j] += ti * tj
    else:
        # X^(2 q^HFEDegI): the two exponents coincide, so (unlike the
        # branch above) this is a "doubled"/square-type term -- genuinely
        # quadratic, full bilinear expansion, coefficient 1 (monic).
        for i in range(n):
            ti = theta[i] ** exp_i
            for j in range(n):
                tj = theta[j] ** exp_i
                Q[i, j] += ti * tj

    # --- "doubled" cross terms X^(2 q^i), i.e. i = j pairs in the double
    # sum defining the HFE polynomial's quadratic terms (see
    # `random_hfe_polynomial`): for every i with i < HFEDegI and i <=
    # HFEDegJ, the polynomial contains a term at exponent q^i + q^i.
    #
    # For q = 2 this exponent equals q^(i+1) -- exactly the "linear term"
    # exponent already handled above -- and the *same* underlying
    # coefficient of F is shared between both conceptual contributions
    # (Sage/the field only stores one coefficient per exponent).
    # For q > 2 this exponent is distinct from every linear exponent, was
    # never read by any loop above, and must be added here as a real
    # quadratic contribution (with its genuine cross terms this time).
    if q != 2:
        for i in range(min(HFEDegI, HFEDegJ + 1)):
            exp_i = q**i
            coef = F[2 * exp_i]
            if coef == 0:
                continue
            for a in range(n):
                ta = theta[a] ** exp_i
                for b in range(n):
                    Q[a, b] += ta * (theta[b] ** exp_i) * coef

    return Q, L

## 3. Add-on transformations: Dragon and Internal Perturbation

Both add-ons start from the plain HFE affine-quadratic form (Q, L) and add
extra terms; neither touches the terms already present in
`hfe_quadratic_form`, so both are written as functions that take an
existing (Q, L) pair and extend it. Dragon only ever adds genuinely
quadratic (cross) terms, so it extends Q (and simply zero-pads L to match
the larger variable set); IP substitutes a *linear* function of x for the
hidden variables z into terms that were already quadratic in z, so its
contribution is also purely quadratic in x and only ever touches Q.

**Dragon** appends $k$ auxiliary variables $Y_0,\dots,Y_{k-1}$ (derived
deterministically from the hash of the message) and couples them to $X$
linearly:

$$Q_{\mathrm{ext}} \ \mathrel{+}= \ \sum_{t=0}^{I} \sum_j L[t,j]\, X^{q^t} Y_j.$$

**Internal Perturbation (IP)** substitutes $r$ hidden variables
$z = M_Z x$ into $F$, without introducing new public variables:

$$Q \ \mathrel{+}=\ \sum_{t=0}^{I}\sum_j M_{\mathrm{bilin}}[t,j]\, X^{q^t} z_j
\ +\ \sum_{i \le j} H_Q[i,j]\, z_i z_j .$$

In [38]:
def add_dragon_terms(Q, L, F, HFEDegI, DragonMat, q):
    """
    Extend an n x n HFE affine-quadratic form (Q, L) (over K =
    F.base_ring()) with the bilinear cross-terms sum_t sum_j
    DragonMat[t,j] * X^(q^t) * Y_j coming from the Dragon hash-embedding
    trick, and return the resulting (Qext, Lext) pair in the extended
    (n + hash_length)-dimensional variable set (x_0,...,x_{n-1},
    Y_0,...,Y_{hash_length-1}). These cross-terms are quadratic (they
    involve both an x and a Y variable), so only Q grows; L is simply
    zero-padded to match the larger variable set.

    DragonMat must have at least HFEDegI + 1 rows; only rows 0..HFEDegI
    are read.
    """
    K = F.base_ring()
    alpha = K.gen()
    n = Q.nrows()
    hash_length = DragonMat.ncols()
    theta = [alpha**i for i in range(n)]

    Qext = Matrix(K, n + hash_length, n + hash_length)
    for i in range(n):
        for j in range(n):
            Qext[i, j] = Q[i, j]
    Lext = vector(K, list(L) + [K.zero()] * hash_length)

    for t in range(HFEDegI + 1):
        exp = q**t
        for j in range(hash_length):
            coef = DragonMat[t, j]
            if coef == 0:
                continue
            for i in range(n):
                Qext[i, n + j] += (theta[i] ** exp) * coef

    return Qext, Lext


def add_internal_perturbation_terms(Q, F, HFEDegI, MZ, MBilin, HQ, q):
    """
    Extend an n x n HFE quadratic form Q in place with the internal-
    perturbation terms coming from substituting z = MZ * x, an auxiliary
    r-dimensional vector of hidden variables:

        sum_{t=0}^{HFEDegI} sum_j MBilin[t,j] * X^(q^t) * z_j
      + sum_{i<=j} HQ[i,j] * z_i * z_j

    Both families are quadratic in x (z is itself linear in x, so
    "X * z" and "z * z" are both degree-2 in x), so only Q is touched --
    IP never introduces a genuinely linear contribution, and does not
    introduce new public variables either, so the result is still n x n.
    Returns Q (also mutated in place).
    """
    K = F.base_ring()
    alpha = K.gen()
    n = Q.nrows()
    r = MZ.nrows()
    theta = [alpha**i for i in range(n)]

    # bilinear X^(q^t) z_j terms
    for t in range(HFEDegI + 1):
        exp = q**t
        for j in range(r):
            coef = MBilin[t, j]
            if coef == 0:
                continue
            row = MZ.row(j)
            for a in range(n):
                ta = theta[a] ** exp
                for b in range(n):
                    if row[b] != 0:
                        Q[a, b] += coef * ta * row[b]

    # quadratic z_i z_j terms
    for i in range(r):
        zi = MZ.row(i)
        for j in range(i, r):
            coef = HQ[i, j]
            if coef == 0:
                continue
            zj = MZ.row(j)
            for a in range(n):
                if zi[a] == 0:
                    continue
                for b in range(n):
                    if zj[b] != 0:
                        Q[a, b] += coef * zi[a] * zj[b]

    return Q


def hfe_dragon_quadratic_form(F, q, n, HFEDegI, HFEDegJ, DragonMat):
    Q, L = hfe_quadratic_form(F, q, n, HFEDegI, HFEDegJ)
    return add_dragon_terms(Q, L, F, HFEDegI, DragonMat, q)


def hfe_ip_quadratic_form(F, q, n, HFEDegI, HFEDegJ, MZ, MBilin, HQ):
    Q, L = hfe_quadratic_form(F, q, n, HFEDegI, HFEDegJ)
    Q = add_internal_perturbation_terms(Q, F, HFEDegI, MZ, MBilin, HQ, q)
    return Q, L


def hfe_ip_dragon_quadratic_form(F, q, n, HFEDegI, HFEDegJ, MZ, MBilin, HQ, DragonMat):
    Q, L = hfe_quadratic_form(F, q, n, HFEDegI, HFEDegJ)
    Q = add_internal_perturbation_terms(Q, F, HFEDegI, MZ, MBilin, HQ, q)
    return add_dragon_terms(Q, L, F, HFEDegI, DragonMat, q)

## 4. Public parameters

All of $q, n, m, r, \mathrm{HFEDegI}, \mathrm{HFEDegJ}, \mathrm{MAX\_SALT}, \mathrm{hash\_length}$
are *public* scheme parameters (known to signer, verifier and attacker
alike) — none of them are secret, so they are bundled into a single
`Params` object that is passed explicitly to every `keygen` / `sign` /
`verify` call below, instead of being read from hidden global variables.
`r`, `MAX_SALT` and `hash_length` are only used by the IP and Dragon add-ons, and default to `None` for plain HFE.

In [39]:
Params = namedtuple("Params", ["q", "n", "m", "HFEDegI", "HFEDegJ", "MAX_SALT", "r", "hash_length"])
Params.__new__.__defaults__ = (None, None, None)  #MAX_SALT, r, hash_length default to None

## 5. Scheme 1 — plain HFE

* **Key generation.** Draw a random HFE polynomial $F$ and its quadratic
  form $Q$, mask it with a secret invertible $S$, and project the equations
  from $n$ down to $m$ with (the first $m$ rows of) a secret invertible
  $T$. The public key is $(\mathrm{cst}_{\mathrm{pub}}, Q_{\mathrm{pub}})$.
* **Signing.** Hash the message to $m$ digits, pad to $n$ digits at random
  if $m<n$ (retrying the padding until $F(X)=U$ has a root), pull the root
  back through $S^{-1}$.
* **Verification.** Evaluate the public quadratic form at the candidate
  signature and check it reproduces the $m$ hash digits.

In [40]:
def hfe_keygen(params):
    q, n, m = params.q, params.n, params.m
    HFEDegI, HFEDegJ = params.HFEDegI, params.HFEDegJ
    if m > n:
        raise ValueError("Need m <= n")

    Fq = GF(q)
    K = GF(q**n, name="alpha")
    Rx = PolynomialRing(K, "X")
    F = random_hfe_polynomial(Rx, q, HFEDegI, HFEDegJ)

    Q, L = hfe_quadratic_form(F, q, n, HFEDegI, HFEDegJ)
    S = random_invertible_matrix(Fq, n)
    T = random_invertible_matrix(Fq, n)

    Kpub = GF(q**m, name="b")
    Tpub = T[:m, :]
    cst_pub = encode_public_coefficient(F[0], Tpub, q, n, Kpub)
    Qpub, Lpub = public_affine_form(Q, L, S, T, q, n, m, Kpub)

    pk = (cst_pub, Qpub, Lpub)
    sk = (F, S.inverse(), T.inverse())
    return pk, sk


def hfe_sign(message, sk, params):
    q, n, m = params.q, params.n, params.m
    F, S_inv, T_inv = sk
    K = F.base_ring()
    alpha = K.gen()
    Fq = GF(q)

    h = hash_to_digits(message, m, q)
    while True:
        if m < n:
            pad = random_vector(Fq, n - m)
            c = vector(Fq, list(h) + list(pad))
        else:
            c = vector(Fq, h)

        U_digits = T_inv * c
        U = sum(U_digits[i] * alpha**i for i in range(n))

        roots = (F - U).roots()
        root = next((rt for rt, mult in roots), None)
        if root is not None:
            break
        if m == n:
            raise RuntimeError("No root found for this HFE instance")

    x = vector(root)
    return x * S_inv


def hfe_verify(pk, message, sig, params):
    q, m = params.q, params.m
    cst_pub, Qpub, Lpub = pk
    h = hash_to_digits(message, m, q)
    value = evaluate_affine_form(Qpub, Lpub, cst_pub, sig)
    return field_element_digits(value, m) == h

## 6. Scheme 2 — HFE-Dragon

The Dragon add-on appends $\mathrm{hash\_length}$ auxiliary variables $Y$
and requires the signer to find a root of $F_h(X) = F(X) + \sum_j h_j
L_j(X) = 0$, where $h$ is the message hash. The variables $Y$ are set equal
to $h$, are **not** masked by $S$ (only the $x$-block is), and are appended
to the public quadratic form; verification checks that the public form
vanishes at $(\sigma, h)$.

In [41]:
def hfe_dragon_keygen(params):
    q, n, m = params.q, params.n, params.m
    HFEDegI, HFEDegJ = params.HFEDegI, params.HFEDegJ
    hash_length = params.hash_length
    if m > n:
        raise ValueError("Need m <= n")

    Fq = GF(q)
    K = GF(q**n, name="alpha")
    Rx = PolynomialRing(K, "X")
    F = random_hfe_polynomial(Rx, q, HFEDegI, HFEDegJ)

    # DragonMat has n rows: only the first HFEDegI are random, the rest
    # (including row HFEDegI, i.e. the coefficient of X^(q^HFEDegI) * Y_j)
    # are zero.
    DragonMat = random_matrix(K, HFEDegI, hash_length).stack(
        zero_matrix(K, n - HFEDegI, hash_length)
    )

    Q, L = hfe_dragon_quadratic_form(F, q, n, HFEDegI, HFEDegJ, DragonMat)
    N = n + hash_length

    Sx = random_invertible_matrix(Fq, n)
    S = block_diagonal_matrix(Sx, identity_matrix(Fq, hash_length))
    T = random_invertible_matrix(Fq, n)

    Kpub = GF(q**m, name="b")
    Tpub = T[:m, :]
    cst_pub = encode_public_coefficient(F[0], Tpub, q, n, Kpub)
    Qpub, Lpub = public_affine_form(Q, L, S, T, q, n, m, Kpub)

    pk = (cst_pub, Qpub, Lpub)
    # S is block-diagonal, so the top-left n x n block of S^{-1} is Sx^{-1}.
    sk = (F, HFEDegI, DragonMat, Sx.inverse(), T.inverse())
    return pk, sk


def hfe_dragon_sign(message, sk, params):
    q, MAX_SALT = params.q, params.MAX_SALT
    F, HFEDegI, DragonMat, S_inv, T_inv = sk
    K = F.base_ring()
    Rx = F.parent()
    X = Rx.gen()
    hash_length = DragonMat.ncols()

    for salt in range(MAX_SALT):
        h = hash_to_digits((message, salt), hash_length, q)

        Fh = F
        for j in range(hash_length):
            if h[j] == 0:
                continue
            yj = K(h[j])
            for k in range(HFEDegI + 1):
                coef = DragonMat[k, j]
                if coef != 0:
                    Fh += yj * coef * X**(q**k)

        roots = Fh.roots()
        root = next((rt for rt, mult in roots if rt != 0), None)

        if root is not None:
            x = vector(root)
            return x * S_inv

    raise RuntimeError("Unable to sign that message")


def hfe_dragon_verify(pk, message, sig, params):
    q, m, MAX_SALT = params.q, params.m, params.MAX_SALT
    hash_length = params.hash_length
    cst_pub, Qpub, Lpub = pk
    for counter in range(MAX_SALT):
        h = hash_to_digits((message, counter), hash_length, q)

        v = vector(GF(q), list(sig) + h)
        value = evaluate_affine_form(Qpub, Lpub, cst_pub, v)

        if field_element_digits(value, m) == [0] * m:
            return True

    return False

## 7. Scheme 3 — HFE-IP

The internal-perturbation add-on hides $r$ extra variables $z = M_Z x$
inside $F$. Signing repeatedly *guesses* $z \in \mathbb F_q^r$, builds the
corresponding polynomial $F_z$, tries to sign with it using the plain-HFE
signer, and accepts the first guess whose recovered preimage $x$ actually
satisfies $M_Z x = z$. Verification is **identical** to plain HFE (IP only
changes key generation and signing, not the public equation), so
`hfe_ip_verify` simply delegates to `hfe_verify`.

In [42]:
def hfe_ip_keygen(params):
    q, n, m, r = params.q, params.n, params.m, params.r
    HFEDegI, HFEDegJ = params.HFEDegI, params.HFEDegJ
    if m > n:
        raise ValueError("Need m <= n")

    Fq = GF(q)
    K = GF(q**n, name="alpha")
    Rx = PolynomialRing(K, "X")
    F = random_hfe_polynomial(Rx, q, HFEDegI, HFEDegJ)

    MZ = random_matrix(Fq, r, n, algorithm="echelonizable", rank=r)

    MBilin = Matrix(K, HFEDegI + 1, r)
    for i in range(HFEDegI + 1):
        for j in range(r):
            MBilin[i, j] = K.random_element()

    HQ = Matrix(K, r, r)
    for i in range(r):
        for j in range(i, r):
            HQ[i, j] = K.random_element()

    Q, L = hfe_ip_quadratic_form(F, q, n, HFEDegI, HFEDegJ, MZ, MBilin, HQ)

    S = random_invertible_matrix(Fq, n)
    T = random_invertible_matrix(Fq, n)

    Kpub = GF(q**m, name="b")
    Tpub = T[:m, :]
    cst_pub = encode_public_coefficient(F[0], Tpub, q, n, Kpub)
    Qpub, Lpub = public_affine_form(Q, L, S, T, q, n, m, Kpub)

    pk = (cst_pub, Qpub, Lpub)
    sk = (F, HFEDegI, MZ, MBilin, HQ, S.inverse(), T.inverse())
    return pk, sk


def hfe_ip_sign(message, sk, params):
    q = params.q
    F, HFEDegI, MZ, MBilin, HQ, S_inv, T_inv = sk
    K = F.base_ring()
    Rx = F.parent()
    X = Rx.gen()
    Fq = GF(q)
    r = MZ.nrows()

    while True:
        z = random_vector(Fq, r)

        Fz = F
        for i in range(r):
            if z[i] == 0:
                continue
            zi = K(z[i])
            for k in range(HFEDegI + 1):
                coef = MBilin[k, i]
                if coef != 0:
                    Fz += zi * coef * X**(q**k)

        cst = K.zero()
        for i in range(r):
            cst += K(z[i])**2 * HQ[i, i]
            for j in range(i + 1, r):
                cst += K(z[i]) * K(z[j]) * HQ[i, j]
        Fz += cst

        try:
            sigma = hfe_sign(message, (Fz, S_inv, T_inv), params)
        except RuntimeError:
            continue

        x = sigma * S_inv.inverse()
        if MZ * x == z:
            return sigma


def hfe_ip_verify(pk, message, sig, params):
    """Internal perturbation does not change the verification equation."""
    return hfe_verify(pk, message, sig, params)

## 8. Scheme 4 — HFE-IP-Dragon

The composition of both add-ons: the message hash first fixes the Dragon
variables $Y$ (giving $F_Y = F + \sum_j Y_j L_j$), then the signer guesses
$z$ as in HFE-IP and looks for a **nonzero root** of $F_{Y,z}(X) = 0$
directly (rather than delegating to the plain-HFE hash-matching signer, as
HFE-IP does). Verification is identical to HFE-Dragon's.

In [43]:
def hfe_ip_dragon_keygen(params):
    q, n, m, r = params.q, params.n, params.m, params.r
    HFEDegI, HFEDegJ = params.HFEDegI, params.HFEDegJ
    hash_length = params.hash_length
    if m > n:
        raise ValueError("Need m <= n")

    Fq = GF(q)
    K = GF(q**n, name="alpha")
    Rx = PolynomialRing(K, "X")
    F = random_hfe_polynomial(Rx, q, HFEDegI, HFEDegJ)

    MZ = random_matrix(Fq, r, n, algorithm="echelonizable", rank=r)

    MBilin = Matrix(K, HFEDegI + 1, r)
    for i in range(HFEDegI + 1):
        for j in range(r):
            MBilin[i, j] = K.random_element()

    HQ = Matrix(K, r, r)
    for i in range(r):
        for j in range(i, r):
            HQ[i, j] = K.random_element()

    # Unlike HFE-Dragon's L, DragonL here has exactly HFEDegI+1 rows, all
    # drawn at random (no forced-zero row).
    DragonL = Matrix(K, HFEDegI + 1, hash_length)
    for i in range(HFEDegI + 1):
        for j in range(hash_length):
            DragonL[i, j] = K.random_element()

    Q, L = hfe_ip_dragon_quadratic_form(F, q, n, HFEDegI, HFEDegJ, MZ, MBilin, HQ, DragonL)
    N = n + hash_length

    Sx = random_invertible_matrix(Fq, n)
    S = block_diagonal_matrix(Sx, identity_matrix(Fq, hash_length))
    T = random_invertible_matrix(Fq, n)

    Kpub = GF(q**m, name="b")
    Tpub = T[:m, :]
    cst_pub = encode_public_coefficient(F[0], Tpub, q, n, Kpub)
    Qpub, Lpub = public_affine_form(Q, L, S, T, q, n, m, Kpub)

    pk = (cst_pub, Qpub, Lpub)
    sk = (F, HFEDegI, MZ, MBilin, HQ, DragonL, Sx.inverse(), T.inverse())
    return pk, sk


def hfe_ip_dragon_sign(message, sk, params):
    q, MAX_SALT = params.q, params.MAX_SALT
    F, HFEDegI, MZ, MBilin, HQ, DragonL, S_inv, T_inv = sk
    K = F.base_ring()
    Rx = F.parent()
    X = Rx.gen()
    Fq = GF(q)
    r = MZ.nrows()
    hash_length = DragonL.ncols()

    for salt in range(MAX_SALT):
        Y = hash_to_digits((message,salt), hash_length, q)
        FY = F
        for j in range(hash_length):
            if Y[j] == 0:
                continue
            yj = K(Y[j])
            for k in range(HFEDegI + 1):
                coef = DragonL[k, j]
                if coef != 0:
                    FY += yj * coef * X**(q**k)    
                    
        for z in Fq^r:            
            Fz = FY
            for i in range(r):
                if z[i] == 0:
                    continue
                zi = K(z[i])
                for k in range(HFEDegI + 1):
                    coef = MBilin[k, i]
                    if coef != 0:
                        Fz += zi * coef * X**(q**k)

            cst = K.zero()
            for i in range(r):
                cst += K(z[i])**2 * HQ[i, i]
                for j in range(i + 1, r):
                    cst += K(z[i]) * K(z[j]) * HQ[i, j]
            Fz += cst

            roots = Fz.roots()
            x_secret = next((rt for rt, mult in roots if rt != 0), None)
            if x_secret is None:
                continue

            x_vec = vector(x_secret)
            if MZ * x_vec != z:
                continue

            return x_vec * S_inv
    raise RuntimeError("Unable to sign that message")


def hfe_ip_dragon_verify(pk, message, sig, params):
    """Same verification equation as HFE-Dragon."""
    return hfe_dragon_verify(pk, message, sig, params)

## 9. Optional: exporting the MQ system as explicit equations

For small toy parameters it can be useful to see (or feed to an external
algebraic solver) the public system as *explicit* multivariate quadratic
polynomials over $\mathbb F_q$, rather than as a matrix over $K$. This
single function replaces the four near-duplicate helpers of the original
code (one pair for the plain case, one pair for the Dragon-extended case);
it works for either simply by passing a wider matrix.

In [44]:
def quadratic_form_to_gfq_system(Q, L, q, n):
    """
    Expand an N x N quadratic form Q and length-N linear part L, both over
    K = GF(q^n), into an explicit system of n multivariate quadratic
    polynomials over GF(q) in N variables x0,...,x_{N-1}: the k-th
    polynomial collects the k-th GF(q)-digit (power-basis coordinate) of
    every entry of Q and L.

    Works uniformly for plain HFE (N = n) and for Dragon-extended forms
    (N = n + hash_length).
    """
    N = Q.nrows()
    quad_coeffs = [Matrix(GF(q), N, N) for _ in range(n)]
    lin_coeffs = [vector(GF(q), N) for _ in range(n)]
    for i in range(N):
        digits = field_element_digits(L[i], n)
        for k in range(n):
            lin_coeffs[k][i] = digits[k]
        for j in range(N):
            digits = field_element_digits(Q[i, j], n)
            for k in range(n):
                quad_coeffs[k][i, j] = digits[k]

    R = PolynomialRing(GF(q), N, names=[f"x{i}" for i in range(N)])
    xs = R.gens()

    equations = []
    for k in range(n):
        f = R.zero()
        for i in range(N):
            if lin_coeffs[k][i] != 0:
                f += lin_coeffs[k][i] * xs[i]
            for j in range(N):
                c = quad_coeffs[k][i, j]
                if c != 0:
                    f += c * xs[i] * xs[j]
        equations.append(f)
    return equations

## 10. Demonstration

`run_scheme_demo` generates a keypair, signs a handful of random messages
and checks verification succeeds, then checks that a tampered signature is
(almost always) rejected. It is run below once for $q = 2$ — with
parameters matching the scale of the original code — and once for a prime
$q > 2$, with deliberately tiny toy parameters chosen only to demonstrate
correctness quickly and cheaply. **None of these parameter sets carry any
security guarantee.**

In [45]:
def run_scheme_demo(name, keygen, sign, verify, params, trials=3, message_len=64):
    tag = f"q={params.q}, n={params.n}, m={params.m}"
    if params.r is not None:
        tag += f", r={params.r}"
    if params.hash_length is not None:
        tag += f", hash_length={params.hash_length}"
    print(f"--- {name}  ({tag}) ---")

    pk, sk = keygen(params)

    for t in range(trials):
        message = [randint(0, 1) for _ in range(message_len)]
        sigma = sign(message, sk, params)
        ok = verify(pk, message, sigma, params)
        print(f"  trial {t + 1}: verify(sign(message)) = {ok}")
        if not ok:
            raise AssertionError("A validly generated signature failed to verify!")

    # Negative test: perturbing one coordinate of a valid signature should
    # (with overwhelming probability) make verification fail.
    message = [randint(0, 1) for _ in range(message_len)]
    sigma = list(sign(message, sk, params))
    sigma[0] = sigma[0] + 1
    tampered = vector(GF(params.q), sigma)
    ok = verify(pk, message, tampered, params)
    print(f"  tampered signature accepted = {ok}  (expected: False)")
    print()

In [53]:
# ---- q = 2, small toy parameters -----------
params_q2       = Params(q=2, n=80, m=64, HFEDegI=2, HFEDegJ=0)
params_q2_ip    = Params(q=2, n=80, m=64, HFEDegI=2, HFEDegJ=0, r=2)
params_q2_drag  = Params(q=2, n=48, m=32, HFEDegI=2, HFEDegJ=0, MAX_SALT = 10, hash_length=64)
params_q2_ipdr  = Params(q=2, n=48, m=32, HFEDegI=2, HFEDegJ=0, MAX_SALT = 10, r=2, hash_length=64)

run_scheme_demo("HFE",           hfe_keygen,           hfe_sign,           hfe_verify,           params_q2)
run_scheme_demo("HFE-IP",        hfe_ip_keygen,        hfe_ip_sign,        hfe_ip_verify,        params_q2_ip)
run_scheme_demo("HFE-Dragon",    hfe_dragon_keygen,    hfe_dragon_sign,    hfe_dragon_verify,    params_q2_drag)
run_scheme_demo("HFE-IP-Dragon", hfe_ip_dragon_keygen, hfe_ip_dragon_sign, hfe_ip_dragon_verify, params_q2_ipdr)

--- HFE  (q=2, n=80, m=64) ---
  trial 1: verify(sign(message)) = True
  trial 2: verify(sign(message)) = True
  trial 3: verify(sign(message)) = True
  tampered signature accepted = False  (expected: False)

--- HFE-IP  (q=2, n=80, m=64, r=2) ---
  trial 1: verify(sign(message)) = True
  trial 2: verify(sign(message)) = True
  trial 3: verify(sign(message)) = True
  tampered signature accepted = False  (expected: False)

--- HFE-Dragon  (q=2, n=48, m=32, hash_length=64) ---
  trial 1: verify(sign(message)) = True
  trial 2: verify(sign(message)) = True
  trial 3: verify(sign(message)) = True
  tampered signature accepted = False  (expected: False)

--- HFE-IP-Dragon  (q=2, n=48, m=32, r=2, hash_length=64) ---
  trial 1: verify(sign(message)) = True
  trial 2: verify(sign(message)) = True
  trial 3: verify(sign(message)) = True
  tampered signature accepted = False  (expected: False)



In [54]:
# ---- q prime, small toy parameters -----------------------------------
q = 5
params_qp       = Params(q=q, n=35, m=28, HFEDegI=1, HFEDegJ=0)
params_qp_ip    = Params(q=q, n=35, m=28, HFEDegI=1, HFEDegJ=0, r=2)
params_qp_drag  = Params(q=q, n=21, m=14, HFEDegI=1, HFEDegJ=0, MAX_SALT = 10, hash_length=28)
params_qp_ipdr  = Params(q=q, n=21, m=14, HFEDegI=1, HFEDegJ=0, MAX_SALT = 10, r=2, hash_length=28)

run_scheme_demo("HFE",           hfe_keygen,           hfe_sign,           hfe_verify,           params_qp)
run_scheme_demo("HFE-IP",        hfe_ip_keygen,        hfe_ip_sign,        hfe_ip_verify,        params_qp_ip)
run_scheme_demo("HFE-Dragon",    hfe_dragon_keygen,    hfe_dragon_sign,    hfe_dragon_verify,    params_qp_drag)
run_scheme_demo("HFE-IP-Dragon", hfe_ip_dragon_keygen, hfe_ip_dragon_sign, hfe_ip_dragon_verify, params_qp_ipdr)

--- HFE  (q=5, n=35, m=28) ---
  trial 1: verify(sign(message)) = True
  trial 2: verify(sign(message)) = True
  trial 3: verify(sign(message)) = True
  tampered signature accepted = False  (expected: False)

--- HFE-IP  (q=5, n=35, m=28, r=2) ---
  trial 1: verify(sign(message)) = True
  trial 2: verify(sign(message)) = True
  trial 3: verify(sign(message)) = True
  tampered signature accepted = False  (expected: False)

--- HFE-Dragon  (q=5, n=21, m=14, hash_length=28) ---
  trial 1: verify(sign(message)) = True
  trial 2: verify(sign(message)) = True
  trial 3: verify(sign(message)) = True
  tampered signature accepted = False  (expected: False)

--- HFE-IP-Dragon  (q=5, n=21, m=14, r=2, hash_length=28) ---
  trial 1: verify(sign(message)) = True
  trial 2: verify(sign(message)) = True
  trial 3: verify(sign(message)) = True
  tampered signature accepted = False  (expected: False)



## Notes and caveats

* **Not constant-time, not hardened.** Root-finding, the IP guess-and-check
  loop, and the toy hash are all written for clarity, not for resistance to
  timing or other side-channel attacks.
* **Toy hash.** `hash_to_digits` is a direct SHA-256-based digit expansion,
  useful for a reproducible proof-of-concept but not a properly
  domain-separated hash-to-$\mathbb F_q^k$ construction.
* **Parameter choice.** None of the parameter sets used in the
  demonstration above (for either $q=2$ or $q>2$) were chosen for security;
  they were chosen to run quickly and illustrate correctness.
* **What changed from $q=2$ to general prime $q$.** Three genuinely
  mathematical fixes were needed, all invisible when $q=2$:
  1. Swapping `GF(2)` for `GF(q)` and bit-shifts `1 << k` for `q^k`
     (cosmetic, but necessary).
  2. In the IP and Dragon add-ons, wherever the original code tested "is
     this coordinate/digit nonzero?" and then added a coefficient
     unconditionally, the coefficient must instead be *scaled* by that
     coordinate's actual value. Both behave identically when $q=2$, since
     every nonzero element of $\mathbb F_2$ equals $1$.
  3. **Linear terms must be tracked separately from quadratic terms.**
     Over $\mathbb F_2$, $x_i^2 = x_i$, so a term linear in $x_i$ can be
     folded into the diagonal of a "quadratic" matrix at no cost, and a
     $q{=}2$-only implementation can get away with a single matrix $Q$
     and no separate linear part. Over $\mathbb F_q$ for $q>2$ this
     identity is false, so this notebook represents every secret and
     public map as an explicit affine-quadratic form $(Q, L, \mathrm{cst})$,
     evaluated as $x^TQx + L{\cdot}x + \mathrm{cst}$
     (`evaluate_affine_form`), and masks/projects $L$ alongside $Q$
     (`public_affine_form`). For $q=2$ this produces the exact same
     signatures and verification behaviour as folding everything into a
     single quadratic matrix — it is a repackaging, not a behavioural
     change — but it is required for correctness when $q>2$: a related,
     more subtle instance of the same issue (an HFE polynomial can contain
     a "doubled" term $X^{2q^i}$, distinct from any linear term, precisely
     when $q>2$) is handled directly inside `hfe_quadratic_form`, with the
     reasoning spelled out in its docstring.
